# GSM8K

This example shows how to evaluate a `genlm.control` model on the GSM8K domain.

* **Task**: Solve grade school math word problems.
* **Data**: GSM8K dataset (Cobbe et al., 2021).

## Setup

First, install the dependencies for this domain. In the root directory, run:    

```bash
pip install -e .
```


## Usage


### Initialize the dataset and evaluator


In [1]:
from genlm.eval.domains.gsm8k import GSM8KDataset, GSM8KEvaluator

dataset = GSM8KDataset.from_hf(
    split="test",
    max_instances=10,
    shuffle=False,
)

print("Instances loaded:", len(dataset))
evaluator = GSM8KEvaluator(tolerance=1e-6)

/opt/miniconda3/envs/gen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instances loaded: 10


### Inspect dataset


In [2]:
first = next(iter(dataset))
print("Instance ID:", first.instance_id)
print("Question:\n", first.question)
print(
    "\nAnswer (with solution):\n",
    first.answer[:200] + "..." if len(first.answer) > 200 else first.answer,
)

Instance ID: 0
Question:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Answer (with solution):
 Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


## Model Adaptor


In [3]:
from genlm.control import PromptedLLM, direct_token_sampler
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.gsm8k import default_prompt_formatter

# Load an LLM
LLM = PromptedLLM.from_name("gpt2", backend="hf", eos_tokens=[b"\n", b"\n\n"])


async def model(instance, output_dir, replicate):
    # Set the prompt for the LLM.
    LLM.prompt_ids = default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=False
    )

    # Use direct token sampler (no constraints for this example)
    sampler = direct_token_sampler(LLM)

    # Generate a response
    sequences = await sampler.smc(
        n_particles=2,
        ess_threshold=0.5,
        max_tokens=100,
    )

    # Return the response as a ModelOutput.
    return ModelOutput(
        responses=[
            ModelResponse(response=sequence, weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

`torch_dtype` is deprecated! Use `dtype` instead!


### Run the evaluation


In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=5,
    n_replicates=1,
    # output_dir="gsm8k_results", #optionally save the results to a directory
)

print(f"\nAverage weighted accuracy: {results['average_weighted_accuracy']:.2%}")
print(f"Number of instances: {results['n_instances']}")

past_key_values should not be None in from_legacy_cache()


Evaluating response:  Red and Blue Dows from the ponds in NatureDward wants to know how much land you buy for ducks and mules in painted ponds.Ducks sold on Amazon.com at usual prices are collected on a weighted average of 40%.
Evaluating response:  Read more >><|endoftext|>Media playback is unsupported on your device Media caption Clinton and Raoul Sanders warned he would be told "no more" in the next Democratic debate
Instance instance_id=0 question="Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?" answer='Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18' metadata={}
Mean weighted accuracy (instance): 0.0
Mean weighted accuracy (total): 0.0

Evaluating response:  15, we are

## Evaluating with Multiple Prompting Strategies

GSM8K supports multiple prompting strategies that can significantly impact model performance. Let's compare different approaches:


In [ ]:
from genlm.control import PromptedLLM
from genlm.eval.domains.gsm8k import (
    chain_of_thought_prompt_formatter,
    direct_answer_prompt_formatter,
    few_shot_prompt_formatter,
)

# Load an LLM
LLM = PromptedLLM.from_name("gpt2", backend="hf", eos_tokens=[b"\n", b"\n\n"])


# Define a factory function to create model adaptors with different prompt formatters
def make_model_adaptor(prompt_formatter):
    """Create a model adaptor with a specific prompt formatter."""

    async def model(instance, output_dir, replicate):
        LLM.prompt_ids = prompt_formatter(
            LLM.model.tokenizer, instance, use_chat_format=False
        )
        sampler = direct_token_sampler(LLM)
        sequences = await sampler.smc(
            n_particles=2,
            ess_threshold=0.5,
            max_tokens=100,
        )
        return ModelOutput(
            responses=[
                ModelResponse(response=sequence, weight=prob)
                for sequence, prob in sequences.decoded_posterior.items()
            ],
        )

    return model


# Create model adaptors for different strategies
strategies = {
    "default": make_model_adaptor(default_prompt_formatter),
    "chain_of_thought": make_model_adaptor(chain_of_thought_prompt_formatter),
    "direct_answer": make_model_adaptor(direct_answer_prompt_formatter),
    "few_shot": make_model_adaptor(few_shot_prompt_formatter),
}

### Run evaluations for each strategy


In [ ]:
# Run evaluation for each strategy
strategy_results = {}

for strategy_name, model_adaptor in strategies.items():
    print(f"\nEvaluating with '{strategy_name}' strategy...")
    results = await run_evaluation(
        dataset=dataset,
        model=model_adaptor,
        evaluator=evaluator,
        max_instances=5,
        n_replicates=1,
        verbosity=0,
        # output_dir=f"gsm8k_results_{strategy_name}",  # Save results separately
    )
    strategy_results[strategy_name] = results
    print(f"  Accuracy: {results['average_weighted_accuracy']:.2%}")

### Compare results across strategies


In [ ]:
import pandas as pd

# Create a comparison table
comparison_data = []
for strategy_name, results in strategy_results.items():
    comparison_data.append(
        {
            "Strategy": strategy_name,
            "Accuracy": results["average_weighted_accuracy"],
            "Instances": results["n_instances"],
        }
    )

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values("Accuracy", ascending=False)
print("\nComparison of Prompting Strategies:")
print(comparison_df.to_string(index=False))

### Alternative: Run a single strategy with custom prompt formatter

You can also create your own custom prompt formatter:


In [ ]:
def custom_prompt_formatter(tokenizer, instance, use_chat_format=False):
    """Example of a custom prompt formatter."""
    # Add your custom prompting logic here
    prompt = f"Math Problem:\n{instance.question}\n\nPlease solve this carefully and show your work."
    return tokenizer.encode(prompt)


# Use it with a model adaptor
custom_model = make_model_adaptor(custom_prompt_formatter)

# Then run evaluation as before
# results = await run_evaluation(
#     dataset=dataset,
#     model=custom_model,
#     evaluator=evaluator,
#     max_instances=5,
#     n_replicates=1,
# )

## References

Karl Cobbe, Vineet Kosaraju, Mohammad Bavarian, Mark Chen, Heewoo Jun, Lukasz Kaiser, Matthias Plappert, Jerry Tworek, Jacob Hilton, Reiichiro Nakano, Christopher Hesse, and John Schulman. Training verifiers to solve math word problems. arXiv preprint arXiv:2110.14168, 2021. URL https://arxiv.org/abs/2110.14168
